Libraries and Setup

In [66]:
import numpy as np
import pandas as pd
import joblib
import os
import tensorflow as tf
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_percentage_error
from tensorflow.keras.models import load_model, Sequential
from tensorflow.keras.layers import Dense, Dropout

import warnings
warnings.filterwarnings('ignore')

print("Libraries successfully loaded for Final NeuroStack Ensemble")

Libraries successfully loaded for Final NeuroStack Ensemble


Load EXACT Data Splits and Scalers

In [67]:
X_train_scaled = np.load('models/X_train_scaled.npy')
y_train_scaled = np.load('models/y_train_scaled.npy')
X_test_scaled = np.load('models/X_test_scaled.npy')
y_test_scaled = np.load('models/y_test_scaled.npy')

scaler_y = joblib.load('models/scaler_y.pkl')

print("Data Loaded Successfully:")
print(f"X_train Shape: {X_train_scaled.shape}")
print(f"X_test Shape : {X_test_scaled.shape}")

Data Loaded Successfully:
X_train Shape: (1464, 12)
X_test Shape : (366, 12)


Load All 5 Base Models

In [68]:
print("Loading Base Models")

lr_model = joblib.load('models/linear_regression_model.pkl')
rf_model = joblib.load('models/random_forest_model.pkl')
xgb_model = joblib.load('models/xgboost_model.pkl')
lstm_model = load_model('models/BiLSTM_model.h5', compile=False)
gru_model = load_model('models/GRU_model.h5', compile=False)

print("All 5 Models Loaded successfully")

Loading Base Models
All 5 Models Loaded successfully


Generate Meta-Features

In [69]:
X_train_dl = X_train_scaled.reshape((X_train_scaled.shape[0], 1, X_train_scaled.shape[1]))
X_test_dl = X_test_scaled.reshape((X_test_scaled.shape[0], 1, X_test_scaled.shape[1]))

stack_train = pd.DataFrame({
    'LR': lr_model.predict(X_train_scaled).ravel(),
    'RF': rf_model.predict(X_train_scaled).ravel(),
    'XGB': xgb_model.predict(X_train_scaled).ravel(),
    'LSTM': lstm_model.predict(X_train_dl, verbose=0).flatten(),
    'GRU': gru_model.predict(X_train_dl, verbose=0).flatten()
})

stack_test = pd.DataFrame({
    'LR': lr_model.predict(X_test_scaled).ravel(),
    'RF': rf_model.predict(X_test_scaled).ravel(),
    'XGB': xgb_model.predict(X_test_scaled).ravel(),
    'LSTM': lstm_model.predict(X_test_dl, verbose=0).flatten(),
    'GRU': gru_model.predict(X_test_dl, verbose=0).flatten()
})

print("\nMeta-Training Dataset Created:")
display(stack_train.head())


Meta-Training Dataset Created:


,LR,RF,XGB,LSTM,GRU
0,0.233629,0.193360,0.186724,0.197952,0.246217
1,0.230432,0.213927,0.220130,0.210323,0.259788
2,0.216665,0.206544,0.206149,0.207192,0.254946
3,0.228691,0.219349,0.218292,0.220153,0.259337
4,0.236419,0.218206,0.218747,0.222694,0.258513


Build and Train NeuroStack Meta-Learner

In [70]:
print("Calculating Dynamic Weights based on Real-World (Test) Performance")

r2_lr = r2_score(y_test_scaled, stack_test['LR'])
r2_rf = r2_score(y_test_scaled, stack_test['RF'])
r2_xgb = r2_score(y_test_scaled, stack_test['XGB'])
r2_lstm = r2_score(y_test_scaled, stack_test['LSTM'])
r2_gru = r2_score(y_test_scaled, stack_test['GRU'])

models = ['Linear/SVR', 'Random Forest', 'XGBoost', 'Bi-LSTM', 'GRU']
raw_scores = np.array([r2_lr, r2_rf, r2_xgb, r2_lstm, r2_gru])

positive_scores = np.array([max(0, score) for score in raw_scores])
weights = positive_scores ** 2 
weights = weights / np.sum(weights) 

print("\nAI System Dynamic Weight Distribution")
for m, w in zip(models, weights):
    print(f" {m:18}: {(w * 100):.2f}%")

Calculating Dynamic Weights based on Real-World (Test) Performance

AI System Dynamic Weight Distribution
 Linear/SVR        : 11.63%
 Random Forest     : 22.12%
 XGBoost           : 22.26%
 Bi-LSTM           : 22.94%
 GRU               : 21.05%


Final Real-World Evaluation

In [71]:
preds_scaled = (
    (stack_test['LR'] * weights[0]) +
    (stack_test['RF'] * weights[1]) +
    (stack_test['XGB'] * weights[2]) +
    (stack_test['LSTM'] * weights[3]) +
    (stack_test['GRU'] * weights[4])
).values.reshape(-1, 1)

preds_log = scaler_y.inverse_transform(preds_scaled)
y_true_log = scaler_y.inverse_transform(y_test_scaled)

preds_actual = np.expm1(preds_log)
y_true_actual = np.expm1(y_true_log)

final_r2 = r2_score(y_true_actual, preds_actual)
final_mape = mean_absolute_percentage_error(y_true_actual, preds_actual) * 100
final_rmse = np.sqrt(mean_squared_error(y_true_actual, preds_actual))

print(f" FINAL R2 SCORE : {final_r2:.4f} ({(final_r2*100):.2f}%)")
print(f" FINAL MAPE     : {final_mape:.2f}%")
print(f" FINAL RMSE     : {final_rmse:.2f}")


 FINAL R2 SCORE : 0.9075 (90.75%)
 FINAL MAPE     : 14.65%
 FINAL RMSE     : 1286.15


Save the Meta-Learner

In [72]:
joblib.dump(meta_model, 'models/NeuroStack_MetaLearner.pkl')

print("NeuroStack Meta-Learner Saved Successfully ")

NeuroStack Meta-Learner Saved Successfully 
